> **Portfolio version.** Cell outputs and workspace-specific connection details have been removed. Configure the environment variables documented in the repository README before running on Databricks.


# Analysis, MLflow and Tableau Serving

Uses `cbda_gold.gold_borough_year_panel` to produce statistical evidence, model runs and Tableau-ready serving tables.

train/test evaluation uses a time-based split


In [ ]:
from functools import reduce

from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType


## 1. Configuration

In [ ]:
catalog = ""
gold_schema = "cbda_gold"
tableau_schema = "cbda_tableau"
write_mode = "overwrite"

GOLD_TABLE_NAME = "gold_borough_year_panel"
LABEL_COL = "crime_rate_per_1000"
TRAIN_END_YEAR = 2020
TEST_START_YEAR = 2021

if catalog:
    gold_ns = f"`{catalog}`.`{gold_schema}`"
    tableau_ns = f"`{catalog}`.`{tableau_schema}`"
else:
    gold_ns = f"`{gold_schema}`"
    tableau_ns = f"`{tableau_schema}`"

gold_table = f"{gold_ns}.`{GOLD_TABLE_NAME}`"

print(f"Gold table: {gold_table}")
print(f"Tableau namespace: {tableau_ns}")
print(f"Train/Test split: train <= {TRAIN_END_YEAR}, test >= {TEST_START_YEAR}")


In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {tableau_ns}")


## 2. Load Gold Table

In [ ]:
gold = spark.table(gold_table)

row_count = gold.count()
borough_count = gold.select("borough_key").distinct().count()
year_count = gold.select("financial_year").distinct().count()
duplicate_count = (
    gold.groupBy("borough_key", "financial_year")
    .count()
    .where(F.col("count") > 1)
    .count()
)

print(f"Rows: {row_count}")
print(f"Boroughs: {borough_count}")
print(f"Financial years: {year_count}")
print(f"Duplicate borough-year rows: {duplicate_count}")

if duplicate_count > 0:
    raise ValueError("Gold table contains duplicate borough-year keys.")

gold.select(
    F.count(F.lit(1)).alias("row_count"),
    F.countDistinct("borough_key").alias("borough_count"),
    F.countDistinct("financial_year").alias("financial_year_count"),
    F.sum(F.when(F.col("flytipping_rate_per_1000").isNull(), 1).otherwise(0)).alias("null_flytipping_rate_rows"),
).display()

## 3. Helper Functions

In [ ]:
def table_name(namespace: str, table: str) -> str:
    return f"{namespace}.`{table}`"


def save_table(df, table: str):
    full_name = table_name(tableau_ns, table)
    (
        df.write
        .format("delta")
        .mode(write_mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {full_name}: rows={df.count()}, cols={len(df.columns)}")
    return full_name


def safe_corr(df, feature: str, label: str = LABEL_COL):
    complete = df.select(feature, label).where(F.col(feature).isNotNull() & F.col(label).isNotNull())
    n = complete.count()
    corr = complete.stat.corr(feature, label) if n > 1 else None
    return corr, n


## 4. Time Summary Tables

In [ ]:
borough_profile = (
    gold.groupBy("borough_key", "borough_name")
    .agg(
        F.count(F.lit(1)).alias("year_count"),
        F.sum("crime_count").alias("total_crimes_2011_2024"),
        F.avg("crime_rate_per_1000").alias("avg_crime_rate_per_1000"),
        F.max("crime_rate_per_1000").alias("max_crime_rate_per_1000"),
        F.sum("flytipping_incidents").alias("total_flytipping_incidents_2011_2024"),
        F.avg("flytipping_rate_per_1000").alias("avg_flytipping_rate_per_1000"),
        F.max("flytipping_rate_per_1000").alias("max_flytipping_rate_per_1000"),
        F.avg("population_mid_year").alias("avg_population_mid_year"),
        F.avg("median_income_gbp").alias("avg_median_income_gbp"),
        F.avg("unemployment_rate_est").alias("avg_unemployment_rate_est"),
        F.avg("csi_overall_score").alias("csi_overall_score"),
        F.avg("living_environment_average_score").alias("living_environment_average_score"),
        F.avg("income_average_score").alias("income_average_score"),
    )
)

w_crime = Window.orderBy(F.desc("avg_crime_rate_per_1000"))
w_fly = Window.orderBy(F.desc("avg_flytipping_rate_per_1000"))
borough_profile = (
    borough_profile
    .withColumn("crime_rate_rank", F.dense_rank().over(w_crime))
    .withColumn("flytipping_rate_rank", F.dense_rank().over(w_fly))
)

year_summary = (
    gold.groupBy("financial_year", "fy_start_year", "pre_covid_period", "covid_period", "post_covid_period")
    .agg(
        F.countDistinct("borough_key").alias("borough_count"),
        F.sum("crime_count").alias("total_crime_count"),
        F.avg("crime_rate_per_1000").alias("avg_crime_rate_per_1000"),
        F.sum("flytipping_incidents").alias("total_flytipping_incidents"),
        F.avg("flytipping_rate_per_1000").alias("avg_flytipping_rate_per_1000"),
        F.avg("median_income_gbp").alias("avg_median_income_gbp"),
        F.avg("unemployment_rate_est").alias("avg_unemployment_rate_est"),
    )
    .orderBy("fy_start_year")
)

save_table(gold, "tableau_borough_year_panel")
save_table(borough_profile, "tableau_borough_profile_panel")
save_table(year_summary, "tableau_year_summary_panel")

display(borough_profile.orderBy("crime_rate_rank"))
display(year_summary)


## 5. Correlation 

In [ ]:
FEATURE_GROUPS = {
    "flytipping_rate_per_1000": "Visible disorder",
    "median_income_gbp": "Annual socioeconomic control",
    "mean_income_gbp": "Annual socioeconomic control",
    "unemployment_rate_est": "Annual socioeconomic control",
    "csi_overall_score": "Static civic context",
    "living_environment_average_score": "Static deprivation context",
    "income_average_score": "Static deprivation context",
    "education_skills_and_training_average_score": "Static deprivation context",
    "health_deprivation_and_disability_average_score": "Static deprivation context",
    "crime_average_score": "Static deprivation context",
    "barriers_to_housing_and_services_average_score": "Static deprivation context",
}

correlation_rows = []
for feature, group in FEATURE_GROUPS.items():
    if feature in gold.columns:
        corr, n_complete = safe_corr(gold, feature)
        direction = None
        if corr is not None:
            if corr > 0:
                direction = "positive"
            elif corr < 0:
                direction = "negative"
            else:
                direction = "zero"
        correlation_rows.append((
            feature,
            group,
            "pearson",
            float(corr) if corr is not None else None,
            abs(float(corr)) if corr is not None else None,
            direction,
            int(n_complete),
        ))

correlation_schema = StructType([
    StructField("feature_name", StringType(), False),
    StructField("feature_group", StringType(), False),
    StructField("method", StringType(), False),
    StructField("correlation_with_crime_rate", DoubleType(), True),
    StructField("abs_correlation", DoubleType(), True),
    StructField("relationship_direction", StringType(), True),
    StructField("n_complete", LongType(), False),
])

correlation_df = spark.createDataFrame(correlation_rows, correlation_schema).orderBy(F.desc("abs_correlation"))
save_table(correlation_df, "tableau_correlation_panel")

feature_value_frames = []
for feature, group in FEATURE_GROUPS.items():
    if feature in gold.columns:
        feature_value_frames.append(
            gold.select(
                "borough_key",
                "borough_name",
                "financial_year",
                "fy_start_year",
                F.lit(feature).alias("feature_name"),
                F.lit(group).alias("feature_group"),
                F.col(feature).cast("double").alias("feature_value"),
                F.col(LABEL_COL).cast("double").alias("crime_rate_per_1000"),
            )
        )

feature_values_long = reduce(lambda left, right: left.unionByName(right), feature_value_frames)
save_table(feature_values_long, "tableau_feature_values_long_panel")

display(correlation_df)


## 6. Dataset

The split avoids random leakage

- train: `fy_start_year <= 2020`
- test: `fy_start_year >= 2021`


In [ ]:
BASELINE_FEATURES = ["flytipping_rate_per_1000"]
PANEL_FEATURES = [
    "flytipping_rate_per_1000",
    "median_income_gbp",
    "unemployment_rate_est",
    "csi_overall_score",
    "living_environment_average_score",
    "income_average_score",
    "covid_period_num",
    "post_covid_period_num",
]

model_gold = (
    gold
    .withColumn("covid_period_num", F.col("covid_period").cast("double"))
    .withColumn("post_covid_period_num", F.col("post_covid_period").cast("double"))
)

model_keys = ["borough_key", "borough_name", "financial_year", "fy_start_year"]
required_cols = [LABEL_COL] + PANEL_FEATURES
model_df = model_gold.select(*model_keys, *required_cols).na.drop(subset=required_cols)

train_df = model_df.where(F.col("fy_start_year") <= TRAIN_END_YEAR)
test_df = model_df.where(F.col("fy_start_year") >= TEST_START_YEAR)

print(f"Model rows after dropping null feature/label rows: {model_df.count()}")
print(f"Train rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

display(model_df.orderBy("borough_name", "fy_start_year"))

In [ ]:
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

try:
    from scipy import stats

    def two_sided_p_value(t_value, df_resid):
        return float(2 * stats.t.sf(abs(t_value), df_resid))
except Exception:
    import math

    def two_sided_p_value(t_value, df_resid):
        return float(math.erfc(abs(t_value) / math.sqrt(2)))


def fit_ols_evidence(pdf, model_name, features):
    cols = [LABEL_COL] + features
    data = pdf[cols].dropna().copy()

    usable_features = []
    dropped_features = []

    for feature in features:
        if data[feature].astype(float).nunique(dropna=True) > 1:
            usable_features.append(feature)
        else:
            dropped_features.append(feature)

    X_raw = data[usable_features].astype(float).to_numpy()
    y = data[LABEL_COL].astype(float).to_numpy()

    X = np.column_stack([np.ones(len(data)), X_raw])
    feature_names = ["intercept"] + usable_features

    beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)

    y_hat = X @ beta
    residuals = y - y_hat

    n_obs = int(X.shape[0])
    n_params = int(X.shape[1])
    df_resid = int(max(n_obs - rank, 1))

    sigma2 = float((residuals @ residuals) / df_resid)
    xtx_inv = np.linalg.pinv(X.T @ X)
    std_errors = np.sqrt(np.diag(xtx_inv) * sigma2)

    rows = []
    for feature_name, coefficient, std_error in zip(feature_names, beta, std_errors):
        if feature_name == "intercept":
            continue

        if std_error == 0 or np.isnan(std_error):
            t_stat = None
            p_value = None
        else:
            t_stat = float(coefficient / std_error)
            p_value = two_sided_p_value(t_stat, df_resid)

        rows.append((
            model_name,
            feature_name,
            float(coefficient),
            float(std_error),
            t_stat,
            p_value,
            n_obs,
            df_resid,
            int(rank),
            n_params,
            ",".join(usable_features),
            ",".join(dropped_features),
        ))

    return rows


ols_train_pdf = train_df.toPandas()

ols_rows = []
ols_rows += fit_ols_evidence(
    ols_train_pdf,
    "baseline_ols_flytipping_only",
    BASELINE_FEATURES,
)
ols_rows += fit_ols_evidence(
    ols_train_pdf,
    "controlled_ols_panel",
    PANEL_FEATURES,
)

ols_schema = StructType([
    StructField("model_name", StringType(), False),
    StructField("feature_name", StringType(), False),
    StructField("coefficient", DoubleType(), True),
    StructField("std_error", DoubleType(), True),
    StructField("t_stat", DoubleType(), True),
    StructField("p_value", DoubleType(), True),
    StructField("n_obs", LongType(), False),
    StructField("df_resid", LongType(), False),
    StructField("model_rank", LongType(), False),
    StructField("n_params", LongType(), False),
    StructField("features_used", StringType(), True),
    StructField("features_dropped", StringType(), True),
])

ols_evidence = spark.createDataFrame(ols_rows, ols_schema)

ols_evidence = (
    ols_evidence
    .withColumn(
        "p_value_label",
        F.when(F.col("p_value") < 0.001, F.lit("<0.001"))
         .otherwise(F.format_string("%.3f", F.col("p_value")))
    )
    .withColumn(
        "significance",
        F.when(F.col("p_value") < 0.001, F.lit("***"))
         .when(F.col("p_value") < 0.01, F.lit("**"))
         .when(F.col("p_value") < 0.05, F.lit("*"))
         .when(F.col("p_value") < 0.1, F.lit("."))
         .otherwise(F.lit(""))
    )
)

save_table(ols_evidence, "tableau_linear_coefficient_evidence_panel")

display(
    ols_evidence
    .orderBy("model_name", F.desc(F.abs(F.col("t_stat"))))
)

## 7. Train Models

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression as SkLinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor as SkRandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

train_pdf = train_df.toPandas()
test_pdf = test_df.toPandas()

metrics_rows = []
importance_rows = []
coefficient_rows = []
prediction_frames = []
models = {}


def fit_and_score(model_name: str, estimator, input_cols):
    X_train = train_pdf[input_cols].values
    y_train = train_pdf[LABEL_COL].values
    X_test = test_pdf[input_cols].values
    y_test = test_pdf[LABEL_COL].values

    estimator.fit(X_train, y_train)
    y_pred = estimator.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(mean_absolute_error(y_test, y_pred))
    r2 = float(r2_score(y_test, y_pred))

    metrics_rows.append((
        model_name,
        "time_based_train_to_2020_test_2021_2023",
        int(len(X_train)),
        int(len(X_test)),
        rmse,
        mae,
        r2,
        ",".join(input_cols),
    ))

    pred_pdf = test_pdf[model_keys].copy()
    pred_pdf["actual_crime_rate_per_1000"] = y_test
    pred_pdf["predicted_crime_rate_per_1000"] = y_pred
    pred_pdf["model_name"] = model_name
    prediction_frames.append(spark.createDataFrame(pred_pdf))

    models[model_name] = (estimator, input_cols)
    return estimator


baseline_model = fit_and_score(
    "baseline_linear_flytipping_only",
    SkLinearRegression(),
    BASELINE_FEATURES,
)

controlled_lr_model = fit_and_score(
    "controlled_linear_panel",
    Ridge(alpha=0.1),
    PANEL_FEATURES,
)

rf_model = fit_and_score(
    "random_forest_panel",
    SkRandomForestRegressor(n_estimators=200, max_depth=5, random_state=42),
    PANEL_FEATURES,
)

for feature, coef in zip(PANEL_FEATURES, controlled_lr_model.coef_.tolist()):
    coefficient_rows.append(("controlled_linear_panel", feature, "coefficient", float(coef)))

for feature, importance in zip(PANEL_FEATURES, rf_model.feature_importances_.tolist()):
    importance_rows.append(("random_forest_panel", feature, "feature_importance", float(importance)))

metrics_schema = StructType([
    StructField("model_name", StringType(), False),
    StructField("split_strategy", StringType(), False),
    StructField("train_rows", LongType(), False),
    StructField("test_rows", LongType(), False),
    StructField("rmse", DoubleType(), False),
    StructField("mae", DoubleType(), False),
    StructField("r2", DoubleType(), False),
    StructField("features", StringType(), False),
])

model_metrics = spark.createDataFrame(metrics_rows, metrics_schema).orderBy(F.desc("r2"))

importance_schema = StructType([
    StructField("model_name", StringType(), False),
    StructField("feature_name", StringType(), False),
    StructField("metric_type", StringType(), False),
    StructField("value", DoubleType(), False),
])

feature_importance = spark.createDataFrame(importance_rows + coefficient_rows, importance_schema)
model_predictions = reduce(lambda left, right: left.unionByName(right), prediction_frames)

save_table(model_metrics, "tableau_model_metrics_panel")
save_table(feature_importance, "tableau_feature_importance_panel")
save_table(model_predictions, "tableau_model_predictions_panel")

display(model_metrics)
display(feature_importance.orderBy("model_name", F.desc("value")))

## 8. Rolling Year Holdout Cross-Validation

a temporal holdout check, not random K-fold, each fold trains on all earlier years and tests on one later year.


In [ ]:
cv_rows = []
cv_years = [2018, 2019, 2020, 2021, 2022, 2023]

model_pdf = model_df.toPandas()

for test_year in cv_years:
    cv_train_pdf = model_pdf[model_pdf["fy_start_year"] < test_year]
    cv_test_pdf = model_pdf[model_pdf["fy_start_year"] == test_year]
    if len(cv_train_pdf) == 0 or len(cv_test_pdf) == 0:
        continue

    for model_name, estimator, input_cols in [
        ("controlled_linear_panel", Ridge(alpha=0.1), PANEL_FEATURES),
        ("random_forest_panel", SkRandomForestRegressor(n_estimators=200, max_depth=5, random_state=42), PANEL_FEATURES),
    ]:
        X_train = cv_train_pdf[input_cols].values
        y_train = cv_train_pdf[LABEL_COL].values
        X_test = cv_test_pdf[input_cols].values
        y_test = cv_test_pdf[LABEL_COL].values

        estimator.fit(X_train, y_train)
        y_pred = estimator.predict(X_test)

        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        mae = float(mean_absolute_error(y_test, y_pred))
        r2 = float(r2_score(y_test, y_pred))

        cv_rows.append((
            model_name,
            int(test_year),
            f"{test_year}-{str(test_year + 1)[-2:]}",
            int(len(X_train)),
            int(len(X_test)),
            rmse,
            mae,
            r2,
        ))

cv_schema = StructType([
    StructField("model_name", StringType(), False),
    StructField("test_fy_start_year", LongType(), False),
    StructField("test_financial_year", StringType(), False),
    StructField("train_rows", LongType(), False),
    StructField("test_rows", LongType(), False),
    StructField("rmse", DoubleType(), False),
    StructField("mae", DoubleType(), False),
    StructField("r2", DoubleType(), False),
])

cross_validation = spark.createDataFrame(cv_rows, cv_schema).orderBy("model_name", "test_fy_start_year")
save_table(cross_validation, "tableau_cross_validation_panel")
display(cross_validation)

## 9. MLflow

In [ ]:
import mlflow

catalog = ""
gold_schema = "cbda_gold"
tableau_schema = "cbda_tableau"
GOLD_TABLE_NAME = "gold_borough_year_panel"
LABEL_COL = "crime_rate_per_1000"

if catalog:
    gold_table = f"`{catalog}`.`{gold_schema}`.`{GOLD_TABLE_NAME}`"
    tableau_ns = f"`{catalog}`.`{tableau_schema}`"
else:
    gold_table = f"`{gold_schema}`.`{GOLD_TABLE_NAME}`"
    tableau_ns = f"`{tableau_schema}`"

experiment_path = "/Shared/cbda_broken_windows"
mlflow.set_experiment(experiment_path)

# metrics
metrics_pdf = spark.table(f"{tableau_ns}.`tableau_model_metrics_panel`").toPandas()
cv_pdf = spark.table(f"{tableau_ns}.`tableau_cross_validation_panel`").toPandas()

# Log 
for _, row in metrics_pdf.iterrows():
    with mlflow.start_run(run_name=row["model_name"]) as run:
        # Params
        mlflow.log_param("model_name", row["model_name"])
        mlflow.log_param("split_strategy", row["split_strategy"])
        mlflow.log_param("features", row["features"])
        mlflow.log_param("train_rows", int(row["train_rows"]))
        mlflow.log_param("test_rows", int(row["test_rows"]))
        mlflow.log_param("label", LABEL_COL)
        mlflow.log_param("gold_table", gold_table)

        # metrics 
        mlflow.log_metric("rmse", row["rmse"])
        mlflow.log_metric("mae", row["mae"])
        mlflow.log_metric("r2", row["r2"])

        # CV metrics
        model_cv = cv_pdf[cv_pdf["model_name"] == row["model_name"]]
        if not model_cv.empty:
            mlflow.log_metric("cv_mean_r2", float(model_cv["r2"].mean()))
            mlflow.log_metric("cv_std_r2", float(model_cv["r2"].std()))
            mlflow.log_metric("cv_mean_rmse", float(model_cv["rmse"].mean()))

        print(f"\u2713 Run '{row['model_name']}' | R\u00b2={row['r2']:.4f} | RMSE={row['rmse']:.2f} | MAE={row['mae']:.2f}")

print(f"\n Experiment: {experiment_path}")
print(f" runs logged")

In [ ]:
output_tables = [
    "tableau_borough_year_panel",
    "tableau_borough_profile_panel",
    "tableau_year_summary_panel",
    "tableau_correlation_panel",
    "tableau_feature_values_long_panel",
    "tableau_model_metrics_panel",
    "tableau_feature_importance_panel",
    "tableau_model_predictions_panel",
    "tableau_cross_validation_panel",
    "tableau_linear_coefficient_evidence_panel",
]

summary_rows = []
for table in output_tables:
    df = spark.table(table_name(tableau_ns, table))
    summary_rows.append((table, df.count(), len(df.columns)))

summary = spark.createDataFrame(summary_rows, ["table_name", "row_count", "column_count"])
display(summary.orderBy("table_name"))
